# Chapter 18: The Tensor Notation

Source orientation: printed pages 328-341; PDF pages 346-359. I inspected the rendered source span privately for structure and terminology: dual bases, the fundamental tensor, reciprocal lattices, the critical lattice of a sphere, general coordinates, Jacobians, and the alternating symbol. This notebook is original teaching material and does not reproduce textbook prose, exercises, screenshots, page crops, or source figures.

## Chapter Goal

Tensor notation is introduced here as a bookkeeping system for geometry that no longer wants to be trapped in rectangular coordinates. A basis can be oblique. Its dual basis records the planes and measurements that recover components. The fundamental tensor records inner products of basis vectors, so it converts contravariant components to covariant components and back. Reciprocal lattices make the same idea visible in crystallography: one lattice of points is paired with another lattice of planes and normals. General coordinates replace fixed Cartesian axes with coordinate curves and level surfaces, and the Jacobian records the local volume scale. The alternating symbol then packages determinants and oriented volume into index notation.

The notebook is visualization-first because tensor notation is easy to misread as mere symbolism. Every symbol below has a picture: arrows for primal and dual bases, ellipses for the metric, point grids for reciprocal lattices, cells for FCC/BCC critical lattices, coordinate mesh surfaces for cylindrical coordinates, and determinant diagrams for the alternating symbol.

## Visualization Storyboard

| Section | Representation | Artifact | Inspection target | Check |
|---|---|---|---|---|
| Dual bases | oblique basis with reciprocal arrows | `dual-basis-metric-tensor.png` | `r^alpha dot r_beta = delta^alpha_beta` | dual basis via inverse and cross-product formulas |
| Fundamental tensor | metric ellipse and component conversion | `covariant-contravariant-components.png` | `u_alpha = g_alpha_beta u^beta` and length forms agree | matrix inverse and reconstruction |
| Reciprocal lattices | direct and reciprocal 2D lattice cells | `reciprocal-lattice-plane-spacing.png` | reciprocal points are normals to rational plane families | `A.T @ B = I` |
| Critical sphere lattice | FCC/BCC cell and nearest vectors | `critical-lattice-fcc-bcc.html` | face-centered cubic and body-centered cubic are reciprocal up to scale | dot-product and determinant checks |
| General coordinates | cylindrical coordinate mesh | `cylindrical-coordinate-metric-jacobian.png` | basis vectors vary and metric becomes `diag(1,r^2,1)` | Jacobian and metric entries |
| Alternating symbol | determinant orientation table | `alternating-symbol-determinant-check.png` | epsilon packages signed volume and cofactors | determinant from Levi-Civita sum |

Matplotlib handles the planar algebra diagrams; Plotly handles the 3D lattice cell because rotating the direct and reciprocal bases is the point. NumPy performs all tensor contractions explicitly so the indices remain inspectable.

In [ ]:
from __future__ import annotations

from itertools import permutations, product
from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, FancyArrowPatch, Polygon
import numpy as np
import plotly.graph_objects as go
from IPython.display import Markdown, display

CHAPTER_NO = 18
HERE = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate the Introduction-to-Geometry book root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, ensure_chapter_artifact_dirs, write_csv, write_json
from utils.course import chapter_by_no

CHAPTER = chapter_by_no(CHAPTER_NO)
DIRS = ensure_chapter_artifact_dirs(CHAPTER_NO, BOOK_ROOT)
FIG = DIRS["figures"]
HTML = DIRS["html"]
CHECKS = DIRS["checks"]
TABLES = DIRS["tables"]

for stale in [FIG / "concept_configuration.svg", FIG / "parameter_experiment.svg", TABLES / "artifact_manifest.csv"]:
    if stale.exists():
        stale.unlink()

COLORS = {
    "ink": "#111827",
    "muted": "#6b7280",
    "blue": "#2563eb",
    "green": "#059669",
    "orange": "#d97706",
    "red": "#dc2626",
    "purple": "#7c3aed",
}

visual_artifacts: list[Path] = []
check_artifacts: list[Path] = []
table_artifacts: list[Path] = []
computed_checks: dict[str, object] = {}

def save_png(fig: plt.Figure, filename: str) -> Path:
    path = FIG / filename
    fig.savefig(path, dpi=165, bbox_inches="tight")
    plt.close(fig)
    visual_artifacts.append(path)
    return path

def display_many(paths: list[Path], width: int = 760) -> None:
    for path in paths:
        display(Markdown(f"### {path.name}"))
        display_artifact(path, width=width)

def rel(path: Path) -> str:
    return path.resolve().relative_to(BOOK_ROOT.resolve()).as_posix()

def arrow2(ax, start, vec, color, label):
    end = np.array(start) + np.array(vec)
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="->", mutation_scale=14, lw=2.0, color=color))
    ax.text(end[0] + 0.04, end[1] + 0.04, label, color=color, fontsize=10)


## 1. Dual Bases And The Fundamental Tensor

Given independent basis vectors `r_1, r_2, r_3`, the dual basis `r^1, r^2, r^3` is defined by `r^alpha dot r_beta = delta^alpha_beta`. The upper index does not mean exponent; it means "belongs to the reciprocal measuring system." A vector can be expanded using contravariant components along the primal basis, while covariant components are the dot products against the primal basis.

The first artifact shows this in a planar projection. The metric tensor `g_alpha_beta = r_alpha dot r_beta` is the matrix of all inner products. Its inverse `g^alpha_beta` is the contravariant tensor. The ellipse is a level set of `u^T G u = 1`, reminding us that in oblique coordinates unit length is not a Euclidean circle in component space.

In [ ]:
R = np.array([
    [1.35, 0.35, 0.20],
    [0.15, 1.10, 0.45],
    [0.10, 0.20, 1.25],
], dtype=float)
dual = np.linalg.inv(R).T
G = R.T @ R
Ginv = np.linalg.inv(G)
J = float(np.linalg.det(R))
dual_from_cross = np.column_stack([
    np.cross(R[:, 1], R[:, 2]) / J,
    np.cross(R[:, 2], R[:, 0]) / J,
    np.cross(R[:, 0], R[:, 1]) / J,
])
identity_error = float(np.max(np.abs(dual.T @ R - np.eye(3))))
cross_formula_error = float(np.max(np.abs(dual - dual_from_cross)))

fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))
ax = axs[0]
arrow2(ax, [0, 0], R[:2, 0], COLORS["blue"], "r_1")
arrow2(ax, [0, 0], R[:2, 1], COLORS["green"], "r_2")
arrow2(ax, [0, 0], dual[:2, 0], COLORS["red"], "r^1")
arrow2(ax, [0, 0], dual[:2, 1], COLORS["orange"], "r^2")
cell = np.array([[0, 0], R[:2, 0], (R[:, 0] + R[:, 1])[:2], R[:2, 1]])
ax.add_patch(Polygon(cell, closed=True, facecolor="#bfdbfe", edgecolor=COLORS["blue"], alpha=0.24))
ax.set_title("primal arrows and dual measuring arrows")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.25)
ax.set_xlabel("x")
ax.set_ylabel("y")

ax = axs[1]
theta = np.linspace(0, 2 * np.pi, 400)
G2 = G[:2, :2]
evals, evecs = np.linalg.eigh(G2)
circle = np.column_stack([np.cos(theta), np.sin(theta)])
ellipse = circle @ np.diag(1 / np.sqrt(evals)) @ evecs.T
ax.plot(ellipse[:, 0], ellipse[:, 1], color=COLORS["purple"], lw=2.4)
ax.axhline(0, color=COLORS["ink"], lw=0.8)
ax.axvline(0, color=COLORS["ink"], lw=0.8)
for v, label, color in [([1, 0], "u^1", COLORS["blue"]), ([0, 1], "u^2", COLORS["green"])]:
    arrow2(ax, [0, 0], v, color, label)
ax.set_title("component-space unit ellipse: u^T G u = 1")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.25)
ax.set_xlabel("contravariant component u^1")
ax.set_ylabel("contravariant component u^2")

dual_path = save_png(fig, "dual-basis-metric-tensor.png")
metric_rows = []
for i in range(3):
    for j in range(3):
        metric_rows.append({"row": i + 1, "column": j + 1, "g_alpha_beta": float(G[i, j]), "g_contravariant": float(Ginv[i, j])})
metric_table = write_csv(TABLES / "fundamental-tensor-matrix.csv", metric_rows)
table_artifacts.append(metric_table)
computed_checks["dual_basis"] = {
    "J": J,
    "dual_identity_error": identity_error,
    "cross_formula_error": cross_formula_error,
    "metric_inverse_error": float(np.max(np.abs(G @ Ginv - np.eye(3)))),
}
assert identity_error < 1e-12
assert cross_formula_error < 1e-12
assert computed_checks["dual_basis"]["metric_inverse_error"] < 1e-12
display_many([dual_path])


## 2. Covariant And Contravariant Components

The same vector has two component languages. Contravariant components `u^alpha` are the coefficients in the expansion `u = sum u^alpha r_alpha`. Covariant components `u_alpha` are inner products `u dot r_alpha`, and the metric converts between them: `u_alpha = g_alpha_beta u^beta` and `u^alpha = g^alpha_beta u_beta`.

The visual below uses a single vector and shows the parallelepiped coordinates that build it. The numerical table records contravariant components, covariant components, reconstructed components, and the equality of length formulas. This is where the tensor notation stops being mysterious: the index position tells you which matrix has already been applied.

In [ ]:
u_contra = np.array([0.85, -0.35, 0.55])
u_vector = R @ u_contra
u_cov = G @ u_contra
u_recovered = Ginv @ u_cov
length_direct = float(u_vector @ u_vector)
length_mixed = float(u_contra @ u_cov)
length_covariant = float(u_cov @ Ginv @ u_cov)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
origin = np.zeros(3)
for vec, label, color in [(R[:, 0], "r_1", COLORS["blue"]), (R[:, 1], "r_2", COLORS["green"]), (R[:, 2], "r_3", COLORS["orange"]), (u_vector, "u", COLORS["red"])]:
    ax.quiver(origin[0], origin[1], origin[2], vec[0], vec[1], vec[2], color=color, linewidth=2.5, arrow_length_ratio=0.08)
    ax.text(vec[0], vec[1], vec[2], label, color=color)
partial1 = u_contra[0] * R[:, 0]
partial2 = partial1 + u_contra[1] * R[:, 1]
chain = np.vstack([origin, partial1, partial2, u_vector])
ax.plot(chain[:, 0], chain[:, 1], chain[:, 2], color=COLORS["muted"], lw=2, ls="--")
ax.set_title("contravariant components build the vector; covariant components measure it")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_box_aspect([1, 1, 1])
component_path = save_png(fig, "covariant-contravariant-components.png")

component_rows = []
for i in range(3):
    component_rows.append({"index": i + 1, "contravariant_u^alpha": float(u_contra[i]), "covariant_u_alpha": float(u_cov[i]), "recovered_contravariant": float(u_recovered[i])})
component_table = write_csv(TABLES / "component-conversion-checks.csv", component_rows)
table_artifacts.append(component_table)
computed_checks["components"] = {
    "reconstruction_error": float(np.linalg.norm(u_recovered - u_contra)),
    "length_direct": length_direct,
    "length_mixed": length_mixed,
    "length_covariant": length_covariant,
}
assert computed_checks["components"]["reconstruction_error"] < 1e-12
assert abs(length_direct - length_mixed) < 1e-12
assert abs(length_direct - length_covariant) < 1e-12
display_many([component_path])


## 3. Reciprocal Lattices

The reciprocal lattice makes duality physical. A direct lattice is generated by a basis `A`; a reciprocal basis is generated by `B = A^{-T}` when we suppress the crystallographic factor of `2*pi`. Points of the reciprocal lattice are normals to families of equally spaced lattice planes in the direct lattice. Large spacings in the direct lattice correspond to short reciprocal vectors.

The 2D artifact uses a skew lattice so the reciprocal cell is visibly rotated and rescaled. The check `A.T @ B = I` is the lattice version of `r^alpha dot r_beta = delta^alpha_beta`.

In [ ]:
A2 = np.array([[1.35, 0.35], [0.25, 1.05]])
B2 = np.linalg.inv(A2).T
pts = np.array([i * A2[:, 0] + j * A2[:, 1] for i in range(-3, 4) for j in range(-3, 4)])
rec_pts = np.array([i * B2[:, 0] + j * B2[:, 1] for i in range(-3, 4) for j in range(-3, 4)])

fig, axs = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, lattice_pts, basis, title, color in [
    (axs[0], pts, A2, "direct lattice", COLORS["blue"]),
    (axs[1], rec_pts, B2, "reciprocal lattice", COLORS["red"]),
]:
    ax.scatter(lattice_pts[:, 0], lattice_pts[:, 1], s=26, color=color, alpha=0.75)
    arrow2(ax, [0, 0], basis[:, 0], COLORS["green"], "basis 1")
    arrow2(ax, [0, 0], basis[:, 1], COLORS["orange"], "basis 2")
    cell = np.array([[0, 0], basis[:, 0], basis[:, 0] + basis[:, 1], basis[:, 1]])
    ax.add_patch(Polygon(cell, closed=True, facecolor="#fef3c7", edgecolor=COLORS["orange"], alpha=0.35))
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.25)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

reciprocal_path = save_png(fig, "reciprocal-lattice-plane-spacing.png")
computed_checks["reciprocal_lattice_2d"] = {
    "identity_error": float(np.max(np.abs(A2.T @ B2 - np.eye(2)))),
    "direct_cell_area": float(abs(np.linalg.det(A2))),
    "reciprocal_cell_area": float(abs(np.linalg.det(B2))),
}
assert computed_checks["reciprocal_lattice_2d"]["identity_error"] < 1e-12
display_many([reciprocal_path])


## 4. FCC, BCC, And The Critical Lattice Of A Sphere

The source span connects reciprocal lattices with face-centered cubic and body-centered cubic geometry. With the unscaled bases `r_1=(0,1,1)`, `r_2=(1,0,1)`, `r_3=(1,1,0)` and reciprocal-style basis `r^1=(-1,1,1)`, `r^2=(1,-1,1)`, `r^3=(1,1,-1)`, the dot products satisfy a scaled dual relation. Dividing one side by two gives the usual reciprocal pairing.

The 3D artifact shows the two cells and nearest lattice vectors. The critical-lattice check records the determinant and the minimum vector length for the FCC basis; the ratio `J/c^3` reaches `1/sqrt(2)` in this normalization, matching the chapter's critical determinant story.

In [ ]:
fcc = np.array([[0, 1, 1], [1, 0, 1], [1, 1, 0]], dtype=float).T
bcc_dual = np.array([[-1, 1, 1], [1, -1, 1], [1, 1, -1]], dtype=float).T
scaled_dual_error = float(np.max(np.abs(bcc_dual.T @ fcc - 2 * np.eye(3))))
J_fcc = abs(float(np.linalg.det(fcc)))
vectors = []
for coeffs in product(range(-1, 2), repeat=3):
    if coeffs != (0, 0, 0):
        vectors.append(fcc @ np.array(coeffs))
vectors = np.array(vectors)
lengths = np.linalg.norm(vectors, axis=1)
min_length = float(lengths.min())
critical_ratio = J_fcc / (min_length ** 3)

fig3 = go.Figure()
def add_cell(basis, name, color, offset=np.zeros(3)):
    verts = [offset + basis @ np.array(c) for c in product([0, 1], repeat=3)]
    edges = []
    for i, a in enumerate(product([0, 1], repeat=3)):
        for j in range(3):
            b = list(a)
            b[j] = 1 - b[j]
            if a[j] == 0:
                edges.append((np.array(a), np.array(b)))
    for e0, e1 in edges:
        p = offset + basis @ e0
        q = offset + basis @ e1
        fig3.add_trace(go.Scatter3d(x=[p[0], q[0]], y=[p[1], q[1]], z=[p[2], q[2]], mode="lines", line=dict(color=color, width=5), name=name, showlegend=False))
    pts_cell = np.array(verts)
    fig3.add_trace(go.Scatter3d(x=pts_cell[:, 0], y=pts_cell[:, 1], z=pts_cell[:, 2], mode="markers", marker=dict(size=4, color=color), name=name))

add_cell(fcc, "FCC basis cell", "#2563eb", np.array([-2.2, 0, 0]))
add_cell(0.5 * bcc_dual, "scaled reciprocal BCC cell", "#dc2626", np.array([2.2, 0, 0]))
nearest = vectors[np.isclose(lengths, min_length)]
for v in nearest[:12]:
    p = np.array([-2.2, 0, 0])
    q = p + v
    fig3.add_trace(go.Scatter3d(x=[p[0], q[0]], y=[p[1], q[1]], z=[p[2], q[2]], mode="lines", line=dict(color="#059669", width=3), showlegend=False))
fig3.update_layout(title="FCC basis and scaled reciprocal BCC basis", width=850, height=650, template="plotly_white", scene=dict(aspectmode="data"))
critical_path = HTML / "critical-lattice-fcc-bcc.html"
if critical_path.exists():
    critical_path.unlink()
fig3.write_html(critical_path, include_plotlyjs="cdn", full_html=True)
visual_artifacts.append(critical_path)

lattice_rows = [
    {"quantity": "scaled_dual_error", "value": scaled_dual_error},
    {"quantity": "J_fcc", "value": J_fcc},
    {"quantity": "min_vector_length", "value": min_length},
    {"quantity": "J_over_c_cubed", "value": critical_ratio},
    {"quantity": "one_over_sqrt2", "value": 1 / math.sqrt(2)},
]
lattice_table = write_csv(TABLES / "fcc-bcc-critical-lattice-checks.csv", lattice_rows)
table_artifacts.append(lattice_table)
computed_checks["critical_lattice"] = {
    "scaled_dual_error": scaled_dual_error,
    "J_over_c_cubed": critical_ratio,
    "critical_ratio_error": abs(critical_ratio - 1 / math.sqrt(2)),
}
assert scaled_dual_error < 1e-12
assert abs(critical_ratio - 1 / math.sqrt(2)) < 1e-12
display_many([critical_path])


## 5. General Coordinates, Metric, And Jacobian

In Cartesian coordinates, basis vectors are constant, orthonormal, and boring in the best possible way. In general coordinates, the basis vectors `r_alpha = partial r / partial u^alpha` vary from point to point. The metric tensor becomes the local rule for measuring displacement: `ds^2 = g_alpha_beta du^alpha du^beta`. The Jacobian gives the volume scale of the coordinate map.

Cylindrical coordinates are the clean example from the source span. The coordinate basis is `(e_r, r e_theta, e_z)`, so the metric is `diag(1, r^2, 1)` and the Jacobian is `r`. The figure shows coordinate surfaces and curves at a selected point.

In [ ]:
r0, theta0, z0 = 1.4, 0.75, 0.7
er = np.array([math.cos(theta0), math.sin(theta0), 0.0])
etheta_scaled = np.array([-r0 * math.sin(theta0), r0 * math.cos(theta0), 0.0])
ez = np.array([0.0, 0.0, 1.0])
G_cyl = np.array([[1, 0, 0], [0, r0 * r0, 0], [0, 0, 1]], dtype=float)
J_cyl = r0

theta = np.linspace(0, 2 * np.pi, 260)
z = np.linspace(-0.5, 1.6, 80)
Theta, Z = np.meshgrid(theta, z)
Rsurf = r0
Xc = Rsurf * np.cos(Theta)
Yc = Rsurf * np.sin(Theta)

fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(121, projection="3d")
ax.plot_surface(Xc, Yc, Z, color="#bfdbfe", alpha=0.32, linewidth=0, shade=False)
for rr in [0.6, 1.0, 1.4, 1.8]:
    ax.plot(rr * np.cos(theta), rr * np.sin(theta), np.full_like(theta, z0), color=COLORS["green"], lw=1.2)
for th in np.linspace(0, 2 * np.pi, 10, endpoint=False):
    ax.plot([0, 2.0 * math.cos(th)], [0, 2.0 * math.sin(th)], [z0, z0], color=COLORS["muted"], lw=0.8)
p = np.array([r0 * math.cos(theta0), r0 * math.sin(theta0), z0])
for vec, color, label in [(er, COLORS["blue"], "r_1"), (etheta_scaled, COLORS["orange"], "r_2"), (ez, COLORS["red"], "r_3")]:
    ax.quiver(p[0], p[1], p[2], vec[0], vec[1], vec[2], color=color, linewidth=2.5, arrow_length_ratio=0.12)
    ax.text(*(p + vec * 1.08), label, color=color)
ax.set_title("cylindrical coordinate basis varies with position")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_box_aspect([1, 1, 0.8])

ax2 = fig.add_subplot(122)
metric_img = ax2.imshow(G_cyl, cmap="Blues")
for i in range(3):
    for j in range(3):
        ax2.text(j, i, f"{G_cyl[i,j]:.2f}", ha="center", va="center")
ax2.set_xticks([0, 1, 2])
ax2.set_yticks([0, 1, 2])
ax2.set_title("metric tensor at the selected point")
fig.colorbar(metric_img, ax=ax2, fraction=0.046)
fig.tight_layout()

cyl_path = save_png(fig, "cylindrical-coordinate-metric-jacobian.png")
computed_checks["general_coordinates"] = {
    "metric_entries": G_cyl.tolist(),
    "jacobian": J_cyl,
    "basis_dot_error": float(np.max(np.abs(np.array([[er @ er, er @ etheta_scaled, er @ ez], [etheta_scaled @ er, etheta_scaled @ etheta_scaled, etheta_scaled @ ez], [ez @ er, ez @ etheta_scaled, ez @ ez]]) - G_cyl))),
}
assert computed_checks["general_coordinates"]["basis_dot_error"] < 1e-12
display_many([cyl_path])


## 6. The Alternating Symbol

The alternating symbol is the determinant written as a contraction. In three dimensions, `epsilon^{alpha beta gamma}` is `1` for even permutations, `-1` for odd permutations, and `0` when any index repeats. This packages oriented area, oriented volume, and cofactor formulas.

The final artifact lists the nonzero permutations and compares the explicit epsilon sum with `numpy.linalg.det`. This is the same orientation logic used earlier for vector triple products and Jacobians, now written in the notation that later differential geometry and tensor calculus can reuse.

In [ ]:
def permutation_sign(p):
    inversions = sum(1 for i in range(len(p)) for j in range(i + 1, len(p)) if p[i] > p[j])
    return -1 if inversions % 2 else 1

def epsilon(i, j, k):
    if len({i, j, k}) < 3:
        return 0
    return permutation_sign((i, j, k))

M = np.array([[1.2, 0.35, -0.25], [0.1, 1.4, 0.55], [0.45, -0.3, 1.1]])
det_eps = 0.0
eps_rows = []
for i, j, k in product(range(3), repeat=3):
    val = epsilon(i, j, k)
    if val:
        term = val * M[0, i] * M[1, j] * M[2, k]
        det_eps += term
        eps_rows.append({"permutation": f"{i+1}{j+1}{k+1}", "epsilon": val, "term": float(term)})
det_np = float(np.linalg.det(M))
eps_table = write_csv(TABLES / "alternating-symbol-terms.csv", eps_rows)
table_artifacts.append(eps_table)

fig, axs = plt.subplots(1, 2, figsize=(12, 5.2))
ax = axs[0]
for row_idx, row in enumerate(eps_rows):
    color = COLORS["blue"] if row["epsilon"] > 0 else COLORS["red"]
    ax.bar(row_idx, row["term"], color=color)
    ax.text(row_idx, row["term"] + 0.02 * np.sign(row["term"]), row["permutation"], ha="center", va="bottom" if row["term"] >= 0 else "top", fontsize=8)
ax.axhline(0, color=COLORS["ink"], lw=0.8)
ax.set_title("epsilon terms in the determinant")
ax.set_ylabel("signed term")
ax.set_xticks([])
ax.grid(axis="y", alpha=0.25)

ax = axs[1]
img = ax.imshow(M, cmap="PuOr")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", fontsize=10)
ax.set_title(f"matrix determinant = {det_np:.4f}")
ax.set_xticks([0, 1, 2])
ax.set_yticks([0, 1, 2])
fig.colorbar(img, ax=ax, fraction=0.046)

epsilon_path = save_png(fig, "alternating-symbol-determinant-check.png")
computed_checks["alternating_symbol"] = {
    "nonzero_epsilon_terms": len(eps_rows),
    "determinant_from_epsilon": float(det_eps),
    "determinant_numpy": det_np,
    "determinant_error": abs(det_eps - det_np),
}
assert computed_checks["alternating_symbol"]["nonzero_epsilon_terms"] == 6
assert computed_checks["alternating_symbol"]["determinant_error"] < 1e-12
display_many([epsilon_path])


## Reading The Chapter As Coordinate Hygiene

The chapter is not asking us to worship indices. It is asking us to keep track of which objects live in the basis, which objects live in the dual basis, and which metric is being used to convert one to the other. That care matters immediately for reciprocal lattices and later for differential geometry of surfaces, where coordinate curves are not usually orthonormal and the metric coefficients vary over the surface.

The notebook's checks form a compact survival kit: dual basis identity, metric inverse identity, covariant/contravariant reconstruction, reciprocal-lattice identity, FCC/BCC scaled duality, cylindrical metric and Jacobian, and determinant from the alternating symbol. If those checks pass, the notation is doing geometry rather than merely decorating formulas.

In [ ]:
manifest_rows = []
for artifact in visual_artifacts:
    manifest_rows.append({"artifact": rel(artifact), "role": "visual", "concept": artifact.stem.replace("-", " ")})
for artifact in table_artifacts:
    manifest_rows.append({"artifact": rel(artifact), "role": "table", "concept": artifact.stem.replace("-", " ")})
manifest_path = write_csv(TABLES / "artifact-manifest.csv", manifest_rows)
table_artifacts.append(manifest_path)

summary_payload = {
    "chapter": CHAPTER_NO,
    "title": CHAPTER["title"],
    "source_printed_pages": CHAPTER["printed"],
    "source_pdf_pages": CHAPTER["pdf"],
    "visuals": [rel(path) for path in visual_artifacts],
    "tables": [rel(path) for path in table_artifacts],
    "checks": computed_checks,
}
summary_path = write_json(CHECKS / "visual-summary.json", summary_payload)
legacy_summary_path = write_json(CHECKS / "visual_summary.json", summary_payload)
check_artifacts.extend([summary_path, legacy_summary_path])

final_sanity = {
    "visual_count": len(visual_artifacts),
    "table_count": len(table_artifacts),
    "dual_identity_error": computed_checks["dual_basis"]["dual_identity_error"],
    "component_reconstruction_error": computed_checks["components"]["reconstruction_error"],
    "reciprocal_identity_error": computed_checks["reciprocal_lattice_2d"]["identity_error"],
    "critical_ratio_error": computed_checks["critical_lattice"]["critical_ratio_error"],
    "determinant_error": computed_checks["alternating_symbol"]["determinant_error"],
}
final_sanity_path = write_json(CHECKS / "final-sanity-summary.json", final_sanity)
check_artifacts.append(final_sanity_path)

assert_artifacts(visual_artifacts + table_artifacts + check_artifacts, min_bytes=80)
assert final_sanity["visual_count"] >= 6
assert final_sanity["dual_identity_error"] < 1e-12
assert final_sanity["component_reconstruction_error"] < 1e-12
assert final_sanity["reciprocal_identity_error"] < 1e-12
assert final_sanity["critical_ratio_error"] < 1e-12
assert final_sanity["determinant_error"] < 1e-12
assert not (FIG / "concept_configuration.svg").exists()
assert not (FIG / "parameter_experiment.svg").exists()
assert not (TABLES / "artifact_manifest.csv").exists()

summary_payload


## Takeaways

- A dual basis is a measuring basis: `r^alpha dot r_beta = delta^alpha_beta`.
- The fundamental tensor `g_alpha_beta` records inner products of the primal basis and converts contravariant components to covariant components.
- The inverse tensor `g^alpha_beta` converts back and gives length formulas in covariant components.
- Reciprocal lattices are dual bases made physical; reciprocal vectors encode normals and spacings of rational plane families.
- The FCC and BCC lattices are reciprocal up to scale, and the FCC basis realizes the critical sphere-lattice ratio in this normalization.
- General coordinates replace constant Cartesian axes with varying coordinate basis vectors; the metric and Jacobian record local measurement.
- The alternating symbol is determinant and oriented volume in index form, preparing the notation for surface geometry and tensor calculus.